# HomerGPT — essai TPU (recette modded) : vitesse, puis mini-run sur les 8 cœurs

In [ ]:
import subprocess, glob, shutil


def run(cmd: str) -> None:
    """Lance une commande en affichant sa sortie ; une erreur arrête le notebook (et la session)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'échec ({p.returncode}) : {cmd}')


for name in ('train_gpt2.py', 'train_tpu.py'):
    shutil.copy(glob.glob(f'/kaggle/input/**/{name}', recursive=True)[0], f'/kaggle/working/{name}')

In [ ]:
run("pip install -q torch==2.9.0 --index-url https://download.pytorch.org/whl/cpu && pip install -q torch_xla==2.9.0 libtpu==0.0.21.1")
run("python -c \"import torch, torch_xla, torch_xla.runtime as xr; print(torch.__version__, torch_xla.__version__, xr.device_type(), xr.global_runtime_device_count()); assert xr.device_type() == 'TPU'\"")

In [ ]:
run('cd /kaggle/working && ARCH=modern python train_tpu.py bench && cat bench.json')

In [ ]:
run('cd /kaggle/working && ARCH=modern FAKE_PRETRAIN=1 MODEL=small OUT=/kaggle/working/essai SESSION=1 HOURS_PRETRAIN=0.15 '
    'TOTAL_PRETRAIN_HOURS=0.15 HOURS_FINETUNE=0.05 EVAL_EVERY_S=120 TOKENS_PER_STEP=131072 python train_tpu.py train')

In [ ]:
run('rm -f /kaggle/working/train_gpt2.py /kaggle/working/train_tpu.py /kaggle/working/essai/*.pt /kaggle/working/essai/checkpoints/*.safetensors; ls -laR /kaggle/working | head -30')